In [1]:
import ee









In [2]:
ee.Initialize(project='mineral-prospectivity-zim')
print("GEE initialized")

GEE initialized


In [3]:
ROI = ee.Geometry.Rectangle([25.24, -22.42, 33.07, -15.61])
print('Region of interest defined.')

Region of interest defined.


In [4]:
START_DATE = '2019-01-01'
END_DATE = '2024-12-31'
BANDS = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ST_B10']
BAND_NAMES = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'TIR']
print(f'Date range: {START_DATE} to {END_DATE}')
print(f'Bands defined.{BANDS}')


Date range: 2019-01-01 to 2024-12-31
Bands defined.['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'ST_B10']


In [5]:
def mask_clouds(image):
    """
    Mask clouds, cloud shadows, and cirrus using Landsat QA_PIXEL,
    combining bit flags with confidence thresholding.
    SR and thermal bands are scaled separately per USGS formulas.

    Reference: USGS, "Landsat Collection 2 Quality Assessment Bands"
    https://www.usgs.gov/landsat-missions/landsat-collection-2-quality-assessment-bands
    """
    qa = image.select('QA_PIXEL')

    dilated_cloud = qa.bitwiseAnd(1 << 1).eq(0)   # Bit 1: Dilated Cloud
    cirrus        = qa.bitwiseAnd(1 << 2).eq(0)   # Bit 2: Cirrus
    cloud         = qa.bitwiseAnd(1 << 3).eq(0)   # Bit 3: Cloud
    cloud_shadow  = qa.bitwiseAnd(1 << 4).eq(0)   # Bit 4: Cloud Shadow

    cloud_confidence  = qa.rightShift(8).bitwiseAnd(3)   # Bits 8-9
    low_cloud_conf    = cloud_confidence.lt(2)

    shadow_confidence = qa.rightShift(10).bitwiseAnd(3)  # Bits 10-11
    low_shadow_conf    = shadow_confidence.lt(2)

    clear_mask = (dilated_cloud.And(cirrus)
                  .And(cloud).And(cloud_shadow)
                  .And(low_cloud_conf).And(low_shadow_conf))

    # SR (optical) bands use one scale/offset; thermal uses a different one — must be split
    SR_BANDS = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    sr = image.select(SR_BANDS).multiply(0.0000275).add(-0.2)
    tir = image.select('ST_B10').multiply(0.00341802).add(149.0)

    scaled = image.addBands(sr, None, True).addBands(tir, None, True)

    return (scaled.updateMask(clear_mask)
                  .select(BANDS)
                  .rename(BAND_NAMES)
                  .copyProperties(image, ['system:time_start']))

In [6]:
def prepare_collection(collection_id):
    """LOAD LANDSAT COLLECTION,FILTER TO OUR RIO AND DATE RANGE APPLY CLOUD MASKING, AND SELECT OUR BANDS """
    return (ee.ImageCollection(collection_id)).filterBounds(ROI).filterDate(START_DATE,END_DATE).map(mask_clouds)
landsat8 =prepare_collection('LANDSAT/LC08/C02/T1_L2')
landsat9=prepare_collection('LANDSAT/LC09/C02/T1_L2')
combined=landsat8.merge(landsat9)
n8=landsat8.size().getInfo()
n9=landsat9.size().getInfo()
print(f'Landsat8 scenes: {n8}')
print(f'Landsat9 scenes: {n9}')
print(f'Total scenes:{ n8+n9}')




Landsat8 scenes: 4836
Landsat9 scenes: 2560
Total scenes:7396


In [7]:
composite= combined.median().clip(ROI)
print('Composite Bands:',composite.bandNames().getInfo())

Composite Bands: ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'TIR']


In [8]:
task=ee.batch.Export.image.toAsset(
    image=composite,
    description='landsat_Zimbabwe_median_2019_2024',
    assetId='projects/mineral-prospectivity-zim/assets/landsat_Zimbabwe_median_2019_2024',
    region=ROI,
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e9

)
#task.start()
print('Export task submitted successfully.')
print('Monitor at https://code.earthengine.google.com/ .')

Export task submitted successfully.
Monitor at https://code.earthengine.google.com/ .


{'Blue_max': 0.38489771484375, 'Blue_mean': 0.053681155789257494, 'Blue_min': -0.011250312500000003, 'Green_max': 0.464643203125, 'Green_mean': 0.0835286146778486, 'Green_min': 0.008437324218749997, 'NIR_max': 0.57582140625, 'NIR_mean': 0.22993252448417315, 'NIR_min': -0.0018536770833333372, 'Red_max': 0.51119017578125, 'Red_mean': 0.11396727864593026, 'Red_min': 0.0033835144779512057, 'SWIR1_max': 0.64331564453125, 'SWIR1_mean': 0.29164785001383586, 'SWIR1_min': 0.0013967039015997033, 'SWIR2_max': 0.538229601531498, 'SWIR2_mean': 0.20752248468846998, 'SWIR2_min': 0.002062993164062495, 'TIR_max': 372.9999407000001, 'TIR_mean': 305.39759702166793, 'TIR_min': 281.85957674}


{'Blue_max': 0.3688571638299851, 'Blue_mean': 0.05019332045978705, 'Blue_min': -0.021475657552083336, 'Green_max': 0.4367332060546875, 'Green_mean': 0.08154504335126392, 'Green_min': 0.010315975260416662, 'NIR_max': 0.5530554847005209, 'NIR_mean': 0.24168030732238782, 'NIR_min': -0.0011921422830930219, 'Red_max': 0.4779171555834573, 'Red_mean': 0.10518624667261803, 'Red_min': 0.0031359084909916, 'SWIR1_max': 0.6310993104784124, 'SWIR1_mean': 0.28044275328127427, 'SWIR1_min': 0.0018667819923199522, 'SWIR2_max': 0.535161606593541, 'SWIR2_mean': 0.19479939974273403, 'SWIR2_min': 0.002394161483759162, 'TIR_max': 372.9999407000001, 'TIR_mean': 307.4581687221098, 'TIR_min': 246.95361716}
